# 00 · The data and the task

**What this shows.** What is in the snapshot: which animals, how many sessions per stage, distribution and
session type, trial counts, and a sanity check that the task is what we think it is (stimulus
distributions, choice contingencies, opto fraction).

**What produced it.** The snapshot — `sc-export-snapshot` turns the Bonsai CSVs listed in `config.yaml`
into one pickle under `<data root>/snapshots/`. Everything downstream (reports, model identification, these
notebooks) loads that pickle; nothing re-reads the CSVs.

**How to run things.**

```bash
sc-export-snapshot --check-only        # which CSVs would be loaded, which are missing
sc-export-snapshot                     # write the snapshot
```

Every command and where it writes: `docs/runs.md`. This notebook needs no run; it only needs the snapshot
(or `SC_NB_SYNTHETIC=1` for the in-memory synthetic experiment that CI uses).

In [ ]:
from nb_setup import *
banner()
exp = load_experiment()
print(f'{len(exp.animals)} animals; cohorts in config:', OPTO_COHORT, '/', SWITCH_COHORT)

## 1 · Animals and sessions

`animal.session_table` is the one-row-per-session view everything selects on: stage, distribution,
session type (`regular` / `opto`, or a type stamped from `config.session_types`), trial counts, accuracy.

In [ ]:
rows = []
for aid, animal in exp.animals.items():
    t = animal.session_table
    rows.append({'animal': aid, 'genotype': animal.metadata.get('genotype', 'unknown'),
                 'sessions': len(t), 'trials': int(t['n_trials'].sum()),
                 'first': t['date'].min(), 'last': t['date'].max(),
                 **{f'n_{k}': int(v) for k, v in t['session_type'].value_counts().items()}})
inventory = pd.DataFrame(rows).fillna(0).sort_values('animal').set_index('animal')
inventory

In [ ]:
# sessions per (distribution × session type), whole experiment
tables = pd.concat([a.session_table.assign(animal=aid) for aid, a in exp.animals.items()], ignore_index=True)
pd.crosstab([tables['distribution']], tables['session_type'], margins=True)

## 2 · One animal's timeline

Accuracy per session in order, coloured by distribution, with session type marked. This is the view the
`select_sessions` presets cut: the expert window is the last half of the Uniform sessions above 70 %.

In [ ]:
aid = sorted(exp.animals)[0]
t = exp.animals[aid].session_table.sort_values('session_idx')
fig, ax = plt.subplots(figsize=(10, 3))
for dist, sub in t.groupby('distribution'):
    ax.scatter(sub['session_idx'], sub['accuracy'], label=dist, s=22)
opto = t[t['session_type'] == 'opto']
ax.scatter(opto['session_idx'], opto['accuracy'], facecolor='none', edgecolor='k', s=60, label='opto session')
ax.axhline(0.7, color='0.6', lw=0.8, ls='--')
ax.set(xlabel='session', ylabel='accuracy', title=f'{aid} — session timeline', ylim=(0.4, 1.0))
ax.legend(fontsize=8, ncol=4, loc='lower right')

## 3 · The task, from the trials

Per distribution: the stimulus histogram (is Hard-A really skewed left?), the fraction of opto trials, and
P(choose B) against stimulus pooled over expert sessions. These are the raw ingredients of every statistic
in `docs/stats_reference.md`.

In [ ]:
from behav_utils import TrialArrays, filter_trials, select_sessions

animal = exp.animals[aid]
dists = [d for d in t['distribution'].unique() if d]
fig, axes = plt.subplots(1, len(dists), figsize=(4 * len(dists), 3), sharey=False)
for ax, dist in zip(np.atleast_1d(axes), dists):
    clean = filter_trials(select_sessions(animal, distribution=dist, min_trials=10))
    if not clean:
        continue
    arr = TrialArrays.from_sessions(clean)
    ax.hist(arr.stimulus, bins=20, color='0.6')
    n_opto = sum(int(s.trials.opto_on.sum()) for s in select_sessions(animal, distribution=dist, min_trials=10))
    ax.set(title=f'{dist}: {arr.n_trials} responded trials, {n_opto} opto', xlabel='stimulus')
plt.tight_layout()

In [ ]:
from behav_utils import compute_stats, PSYCHOMETRIC

rows = []
for dist in dists:
    clean = filter_trials(select_sessions(animal, distribution=dist, min_trials=10))
    if clean:
        s = compute_stats(TrialArrays.from_sessions(clean), ['accuracy', 'side_bias', *PSYCHOMETRIC])
        rows.append(s.rename(dist))
pd.DataFrame(rows).round(3)

## Where next

- `10` — expert behaviour by phase, from an `opto_contrasts` run.
- `11` — adaptation after a distribution switch, from a `switch_adaptation` run.
- `20` / `21` — the opto contrasts.
- `30` / `31` — SBI feature selection and BE/SC model identification.